# Proyecto 1 — Predicción de Consumo Eléctrico con LSTM
## Notebook 1: Análisis Exploratorio de Datos (EDA)

**Objetivo:** Comprender la estructura temporal del consumo eléctrico residencial para informar las decisiones de preprocesamiento y modelado.

**Dataset:** UCI Household Electric Power Consumption  
**Período:** Diciembre 2006 – Noviembre 2010 (~2 millones de registros)

---

## 0. Setup e importaciones

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from utils import load_and_clean, add_temporal_features, FEATURE_COLS, TARGET_COL

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGSIZE = (14, 5)
print('Setup completo.')

## 1. Carga de datos

In [ ]:
CSV_PATH = '../data/raw/household_power_consumption.txt'
df = load_and_clean(CSV_PATH)

print(f'Shape: {df.shape}')
print(f'Período: {df.index.min()} → {df.index.max()}')
print(f'Frecuencia: {pd.infer_freq(df.index)}')
df.head()

In [ ]:
print('=== Estadísticas descriptivas ===')
df.describe().T.style.format('{:.3f}').background_gradient(cmap='Blues')

## 2. Distribuciones univariantes

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.ravel()

for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(df[col], bins=50, edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')

# Ocultar el subplot extra
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de variables — Consumo Eléctrico Residencial', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Serie temporal del target: Global Active Power

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Vista anual
df[TARGET_COL].resample('D').mean().plot(ax=axes[0], linewidth=0.8, color='steelblue')
axes[0].set_title('Consumo diario promedio (Dic 2006 – Nov 2010)', fontsize=12)
axes[0].set_ylabel('kWh')

# Vista semanal (1 mes de ejemplo)
sample = df[TARGET_COL]['2008-01']
sample.plot(ax=axes[1], linewidth=0.8, color='coral')
axes[1].set_title('Consumo horario — Enero 2008', fontsize=12)
axes[1].set_ylabel('kWh')

# Vista diaria (1 semana)
week = df[TARGET_COL]['2008-01-07':'2008-01-13']
week.plot(ax=axes[2], linewidth=1.2, color='teal', marker='.')
axes[2].set_title('Consumo horario — semana del 7 al 13 de Enero 2008', fontsize=12)
axes[2].set_ylabel('kWh')

plt.tight_layout()
plt.show()

## 4. Análisis de estacionalidad

In [ ]:
# Perfil promedio por hora del día
df['hour'] = df.index.hour
df['day_of_week'] = df.index.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly_profile = df.groupby('hour')[TARGET_COL].mean()
hourly_profile.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Consumo promedio por hora del día', fontsize=12)
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('kWh promedio')
axes[0].tick_params(axis='x', rotation=0)

# Perfil por día de la semana
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_profile = df.groupby('day_of_week')[TARGET_COL].mean().reindex(dow_order)
dow_profile.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Consumo promedio por día de la semana', fontsize=12)
axes[1].set_xlabel('Día')
axes[1].set_ylabel('kWh promedio')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

df.drop(columns=['hour', 'day_of_week'], inplace=True)

**Hallazgos clave:**
- Pico de consumo en **horas nocturnas (19-21h)** → cocina y electrodomésticos
- Menor consumo entre las 2h y 6h (horario de sueño)
- El fin de semana muestra mayor consumo diurno (personas en casa)
- Patrón estacional marcado: invierno > verano (calefacción)

## 5. Descomposición estacional (STL)

In [ ]:
# Usamos datos diarios para la descomposición
daily = df[TARGET_COL].resample('D').mean().dropna()
decomp = seasonal_decompose(daily, model='additive', period=7)  # Estacionalidad semanal

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
for ax, component, label in zip(
    axes,
    [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid],
    ['Observado', 'Tendencia', 'Estacionalidad (semanal)', 'Residual']
):
    ax.plot(component, linewidth=0.8)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('kWh')

plt.suptitle('Descomposición Estacional Aditiva — Consumo Diario', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Autocorrelación (ACF y PACF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(df[TARGET_COL].dropna(), lags=72, ax=axes[0], zero=False, alpha=0.05)
axes[0].set_title('ACF — Global Active Power (72 lags horarios)', fontsize=12)
axes[0].set_xlabel('Lag (horas)')

plot_pacf(df[TARGET_COL].dropna(), lags=72, ax=axes[1], zero=False, alpha=0.05, method='ywm')
axes[1].set_title('PACF — Global Active Power (72 lags horarios)', fontsize=12)
axes[1].set_xlabel('Lag (horas)')

plt.tight_layout()
plt.show()

**Interpretación:**
- **ACF:** Autocorrelación significativa en múltiplos de 24h → estacionalidad diaria fuerte
- **PACF:** Los primeros lags (1-3h) tienen correlación directa alta → la LSTM debe capturar dependencias de corto y largo plazo

## 7. Matriz de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.5, square=True, ax=ax,
    vmin=-1, vmax=1
)
ax.set_title('Matriz de Correlación de Pearson — Variables del Dataset', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Feature Engineering — variables temporales

In [ ]:
df_features = add_temporal_features(df)
print(f'Columnas después de feature engineering: {list(df_features.columns)}')
print(f'Shape: {df_features.shape}')
df_features.head()

## 9. Resumen y próximos pasos

| Hallazgo | Implicación para el modelo |
|----------|----------------------------|
| Estacionalidad diaria (24h) y semanal (168h) fuertes | Usar ventana de entrada de **168 horas** (7 días) |
| Alta autocorrelación en los primeros lags | LSTM con al menos 2 capas para capturar dependencias |
| Correlación alta entre `Global_active_power` e `intensity` | Incluir todas las variables como features multivariantes |
| Sin valores faltantes tras preprocesamiento | No se necesita imputation adicional |

**Próximo notebook:** `02_lstm_gru_modeling.ipynb` — Modelado LSTM/GRU end-to-end